# Readout Calibration + ZNE 专项教程

本教程分三层：
1. 高层：`run_auto` 一键开启 readout mitigation + ZNE（`Baihua`）
2. 校准：单独执行 readout calibration，查看 confusion matrices
3. 拆解：手动提交硬件任务，复现 readout mitigation 与 ZNE

In [1]:
from pathlib import Path

import numpy as np

from quantum_hw import QuantumHardwareClient, QuantumCircuit
from quantum_hw.api.backend import Backend
from quantum_hw.calibration.readout import ReadoutCalibrationManager
from quantum_hw.compile.translate import TranslateToBasisGates
from quantum_hw.core.observables import group_observables, append_measurement_basis, pauli_expectation, pauli_support
from quantum_hw.core.readout import mitigate_observable_from_samples
from quantum_hw.core.utils import get_samples
from quantum_hw.core.zne import apply_zne_cz_tripling, zne_linear_extrapolate
from quantum_hw.sim.statevector import simulate_counts

def section(title: str):
    print("\n" + "=" * 18, title, "=" * 18)

## 1) 构造线路与观测量

In [2]:
section("build circuit")
num_qubits = 6
target_qubits = [40, 41, 42, 43, 44, 45] # 根据连接图，手动指定相连的比特
shots = 2048
observables = ["ZZIIII", "IIXXII"]

qc = QuantumCircuit(num_qubits)
qc.h(0)
for i in range(num_qubits - 1):
    qc.cx(i, i + 1)

print("gate count:", len(qc.gates))
print("target_qubits:", target_qubits)
print("observables:", observables)


================== build circuit ==================
gate count: 6
target_qubits: [40, 41, 42, 43, 44, 45]
observables: ['ZZIIII', 'IIXXII']


## 2) 高层：`run_auto`（Baihua）

先跑两个版本：
- baseline：不启用 readout mitigation / ZNE
- rem+zne：同时启用 readout mitigation 和 ZNE

In [3]:
section("run_auto baseline")
client = QuantumHardwareClient()

result_auto_baseline = client.run_auto(
    circuit=qc,
    name="tutorial_readout_zne_baseline",
    num_qubits=num_qubits,
    shots=shots,
    observables=observables,
    prefer_chips="Baihua",
    target_qubits=target_qubits,
    print_true=False,
)

print("baseline observable_values:", result_auto_baseline.observable_values)


================== run_auto baseline ==================
baseline observable_values: {'ZZIIII': 0.896484375, 'IIXXII': 0.029296875}


In [4]:
section("run_auto readout mitigation + zne")
result_auto_rem_zne = client.run_auto(
    circuit=qc,
    name="tutorial_readout_zne_rem",
    num_qubits=num_qubits,
    shots=shots,
    observables=observables,
    prefer_chips="Baihua",
    target_qubits=target_qubits,
    zne=True,
    readout_mitigation=True,
    readout_shots=1024,
    print_true=False,
)

print("rem+zne observable_values (final):", result_auto_rem_zne.observable_values)
print("rem+zne observable_values_raw (1x raw):", result_auto_rem_zne.observable_values_raw)


================== run_auto readout mitigation + zne ==================
rem+zne observable_values (final): {'ZZIIII': 0.9734246220999, 'IIXXII': -0.03253622240448108}
rem+zne observable_values_raw (1x raw): {'ZZIIII': 0.87890625, 'IIXXII': -0.0107421875}


## 3) 单独跑 readout calibration

通过 `ReadoutCalibrationManager` 查看每个量子比特的 confusion matrix（由于上面刚跑过，所以会直接读取缓存文件中的数据）

In [6]:
section("readout calibration only")
chip_name = "Baihua"
backend = Backend(chip_name)
client.chip_name = chip_name
client.chip_backend = backend

repo_root = Path.cwd()
if not (repo_root / "src").exists() and (repo_root.parent / "src").exists():
    repo_root = repo_root.parent
cache_dir = repo_root / "src/quantum_hw/api/.cache"

calibration_manager = ReadoutCalibrationManager(
    cache_dir=cache_dir,
    submit_openqasm_async=client._submit_openqasm_async,
    wait_task=client._wait_task,
    get_task_result=client._get_task_result,
    compact_for_sim=client._compact_for_sim,
    simulate_counts=simulate_counts,  # Baihua 路径不会走到该函数
)

cal = calibration_manager.calibrate_readout(
    target_qubits=target_qubits,
    shots=1024,
    chip_name=chip_name,
    backend=backend,
    print_true=False,
)

per_qubit_confusion = {int(q): np.asarray(m, dtype=float) for q, m in cal.per_qubit_confusion.items()}
print("chip:", chip_name)
print("calibrated qubits:", cal.target_qubits)
for q in cal.target_qubits[:3]:
    print(f"Q{q} confusion matrix:")
    print(per_qubit_confusion[q])


================== readout calibration only ==================
chip: Baihua
calibrated qubits: [40, 41, 42, 43, 44, 45]
Q40 confusion matrix:
[[0.99707031 0.00292969]
 [0.03808594 0.96191406]]
Q41 confusion matrix:
[[0.98242188 0.01757812]
 [0.02636719 0.97363281]]
Q42 confusion matrix:
[[0.99414062 0.00585938]
 [0.05859375 0.94140625]]


## 4) 拆解版：手动提交 Baihua 硬件任务复现 readout + ZNE

流程与 `run_auto` 对齐：
- `group_observables` 分组
- 每组构造测量线路并提交硬件任务（1x 与 3x）
- 先对每个实验校正测量误差
- 然后 1x 与 3x 做线性外推

In [14]:
from quantum_hw.api.quantum_platform import create_provider_runtime
section("manual pipeline on Baihua")
groups = group_observables(observables, num_qubits=num_qubits)

translator = TranslateToBasisGates(
    convert_single_qubit_gate_to_u=True,
    two_qubit_gate_basis=backend.two_qubit_gate_basis,
)

base_qct = client._transpile_with_backend(qc.deepcopy(), backend, target_qubits=target_qubits) # 避免重复编译

obs_raw_1 = {}
obs_raw_3 = {}
obs_rem_1 = {}
obs_rem_3 = {}
manual_task_ids = []

for gi, group in enumerate(groups):
    basis_pattern = group["basis"]

    qct_1 = base_qct.deepcopy()
    append_measurement_basis(qct_1, basis_pattern, target_qubits=target_qubits)
    qct_1 = translator.run(qct_1) # 替换测量基矢，不需修改拓扑连接，仅需确保门已分解成native gate

    qct_3 = apply_zne_cz_tripling(qct_1) # 将一个CZ门换成3个
 
    runtime = create_provider_runtime(provider="quafu", client=client)
    client._active_task_adapter = runtime.task_adapter # 直接使用runtime的task adapter提交任务，以确保计费和资源统计正确
    client._active_resolved_backend = runtime.backend_adapter.resolve_backend(num_qubits=num_qubits, prefer_hardware="Baihua")

    task_id_1 = client._submit_openqasm_async( # 底层提交任务函数
        name=f"manual_readout_zne_g{gi}",
        qasm=qct_1.to_openqasm2,
        shots=shots,
        chip_name=chip_name,
    )
    task_id_3 = client._submit_openqasm_async(
        name=f"manual_readout_zne_g{gi}_zne3",
        qasm=qct_3.to_openqasm2,
        shots=shots,
        chip_name=chip_name,
    )
    manual_task_ids.extend([str(task_id_1), str(task_id_3)])

    status_1 = client._wait_task(task_id_1)
    status_3 = client._wait_task(task_id_3)
    if status_1 != "Finished" or status_3 != "Finished":
        raise RuntimeError(f"manual task failed: g{gi}, status=({status_1}, {status_3})")

    counts_1 = client._get_task_result(task_id_1)["count"]
    counts_3 = client._get_task_result(task_id_3)["count"]
    samples_1 = get_samples(counts_1, num_qubits=num_qubits)
    samples_3 = get_samples(counts_3, num_qubits=num_qubits)

    for obs in group["observables"]:
        support = pauli_support(obs, num_qubits=num_qubits)

        obs_raw_1[obs] = pauli_expectation(samples_1, obs)
        obs_raw_3[obs] = pauli_expectation(samples_3, obs)

        obs_rem_1[obs] = mitigate_observable_from_samples( # 使用confusion matrix校正测量误差
            samples=samples_1,
            support=support,
            per_qubit=per_qubit_confusion,
            target_qubits_group=target_qubits,
            marginal_max_support=10,
)
        obs_rem_3[obs] = mitigate_observable_from_samples(
            samples=samples_3,
            support=support,
            per_qubit=per_qubit_confusion,
            target_qubits_group=target_qubits,
            marginal_max_support=10,
)

obs_zne_raw = {
    obs: zne_linear_extrapolate(obs_raw_1[obs], obs_raw_3[obs])
    for obs in observables
}
obs_zne_rem = {
    obs: zne_linear_extrapolate(obs_rem_1[obs], obs_rem_3[obs])
    for obs in observables
}

print("manual raw(1x):", obs_raw_1)
print("manual zne(raw):", obs_zne_raw)
print("manual rem(1x):", obs_rem_1)
print("manual rem+zne:", obs_zne_rem)


================== manual pipeline on Baihua ==================
manual raw(1x): {'ZZIIII': 0.8974609375, 'IIXXII': 0.037109375}
manual zne(raw): {'ZZIIII': 0.93505859375, 'IIXXII': 0.06787109375}
manual rem(1x): {'ZZIIII': 0.9711927692214393, 'IIXXII': 0.04498985752828502}
manual rem+zne: {'ZZIIII': 1.0061666587269475, 'IIXXII': 0.08228323445872439}


## 5) 与高层结果对比

在真实硬件上，readout mitigation 与 ZNE 的改变量通常更明显。

In [15]:
section("compare high-level vs manual")
for obs in observables:
    high_baseline = float(result_auto_baseline.observable_values[obs])
    high_final = float(result_auto_rem_zne.observable_values[obs])
    man_baseline = float(obs_raw_1[obs])
    man_final = float(obs_zne_rem[obs])

    print(f"{obs}:")
    print(f"  high baseline      = {high_baseline:.6f}")
    print(f"  manual baseline    = {man_baseline:.6f}")
    print(f"  high rem+zne final = {high_final:.6f}")
    print(f"  manual rem+zne     = {man_final:.6f}")


================== compare high-level vs manual ==================
ZZIIII:
  high baseline      = 0.896484
  manual baseline    = 0.897461
  high rem+zne final = 0.973425
  manual rem+zne     = 1.006167
IIXXII:
  high baseline      = 0.029297
  manual baseline    = 0.037109
  high rem+zne final = -0.032536
  manual rem+zne     = 0.082283
